# Voir les bases INSEE traitées

Ce notebook ouvre une base propre à la fois. Il montre quelques lignes, le type des colonnes et le nombre de valeurs manquantes. Il ne modifie pas les fichiers.

Dans la dernière cellule, choisissez `source_population.parquet` ou `source_navettes.parquet`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Trouver le dossier du projet depuis le dossier courant ou son parent.
racine = Path.cwd()
while not (racine / "config.yaml").is_file() and racine.parent != racine:
    dossier_insee = racine / "insee_population_navettes"
    if (dossier_insee / "config.yaml").is_file():
        racine = dossier_insee
        break
    racine = racine.parent

if not (racine / "config.yaml").is_file():
    raise FileNotFoundError("Impossible de trouver le dossier insee_population_navettes.")

dossier_bases = racine / "data" / "processed" / "rail2026"
if not dossier_bases.is_dir():
    raise FileNotFoundError(f"Dossier des bases introuvable : {dossier_bases}")

print("Bases disponibles :")
for fichier in sorted(dossier_bases.glob("source_*.parquet")):
    print("-", fichier.name)

In [ ]:
def afficher_base(nom_fichier, nombre_lignes=5):
    """Afficher simplement le contenu d'une base Parquet propre."""
    chemin = dossier_bases / nom_fichier
    if not chemin.is_file():
        raise FileNotFoundError(f"Base introuvable : {chemin}")

    table = pd.read_parquet(chemin)
    print(f"{len(table):,} lignes et {len(table.columns)} colonnes")
    print("Premières lignes :")
    display(table.head(nombre_lignes))

    print("Type de chaque colonne :")
    display(table.dtypes.rename("type").to_frame())

    manquantes = table.isna().sum()
    manquantes = manquantes[manquantes > 0].sort_values(ascending=False)
    print("Valeurs manquantes par colonne :")
    if manquantes.empty:
        print("Aucune")
    else:
        display(manquantes.rename("nombre").to_frame())

    return table

In [ ]:
# Remplacer le nom par source_navettes.parquet pour voir l'autre base.
nom_base = "source_population.parquet"
table = afficher_base(nom_base)